# 04 · Feature engineering territorial, cliente, TPV y buró

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## Ventanas de transaccionalidad point-in-time

Para ventas $V_{i,d}$ y fecha de decisión $t$, definir ventanas **estrictamente anteriores** a $t$:
$$TPV_{i,w}(t)=\sum_{d=t-w}^{t-1}V_{i,d},\quad N_{i,w}(t)=\sum_{d=t-w}^{t-1}n_{i,d}.$$
Ticket promedio $=TPV/N$ si $N>0$; días activos $=\sum_d\mathbf1(n_{i,d}>0)$. El crecimiento entre ventanas disjuntas de 30 días es $g=(TPV_{30}^{reciente}/TPV_{30}^{previa})-1$, indefinido si el denominador es cero.

In [ ]:
tx=load('plantilla_transacciones.csv')
print('Columnas disponibles:',list(tx.columns))
if not tx.empty:
    display(tx.head(5))
else: print('PENDIENTE: transacciones reales, fecha efectiva y fecha de disponibilidad')

## Volatilidad, concentración y estrés

Con ventas diarias $V_d$:
$$CV_V=\frac{\sqrt{\frac{1}{n-1}\sum_d(V_d-\bar V)^2}}{\bar V},\qquad HHI=\sum_j\left(\frac{V_j}{\sum_l V_l}\right)^2.$$
El $HHI$ puede medirse por terminal, canal o día de semana; especificar dimensión. Maximum drawdown del TPV acumulado requiere definir una serie de nivel positiva; para ventas mensuales, es preferible medir caída pico-a-valle en medias móviles, no aplicar la fórmula de precios sin adaptación.

In [ ]:
if not tx.empty:
    amount_candidates=[c for c in tx if any(s in c.lower() for s in ['amount','monto','valor'])]
    print('Campos de importe candidatos:',amount_candidates)
    if amount_candidates:
        vals=pd.to_numeric(tx[amount_candidates[0]],errors='coerce').dropna()
        print('CV de importes individuales (no CV de ventas diarias):',vals.std()/vals.mean() if len(vals)>1 and vals.mean()!=0 else 'no definido')
else: print('PENDIENTE: volatilidad por ventana y terminal')

## Geografía y riesgo ecológico

Para región $r$, tasa cruda $\hat p_r=y_r/n_r$. Para regiones pequeñas, una aproximación Beta-Binomial:
$$p_r\sim\mathrm{Beta}(\alpha,\beta),\quad E[p_r\mid y_r,n_r]=\frac{y_r+\alpha}{n_r+\alpha+\beta}.$$
Esta suavización no sustituye ajuste por composición de clientes. El indicador regional no debe atribuirse causalmente a un individuo (*falacia ecológica*). Proteger localizaciones sensibles y publicar mapas agregados con umbrales mínimos de conteo.

In [ ]:
geo=load('plantilla_geografia_contexto.csv')
print('Columnas territoriales:',list(geo.columns))
if not geo.empty:display(geo.head())
else: print('PENDIENTE: códigos DANE, vigencia, indicador y periodo de publicación')

## Buró, política y disponibilidad

Score, obligaciones y mora externa deben seleccionarse con un *as-of join*: $x_i(t)=x_{i,s^*}$ donde $s^*=\max\{s:t_{disponibilidad}(x_{i,s})\le t\}$. Distinguir buró en solicitud de refrescos posteriores. Para estrategia de originación, guardar versión y decisión **previas** al desembolso; no reconstruirlas a partir de la política actual.

In [ ]:
bureau=load('plantilla_buro.csv');pol=load('plantilla_politicas.csv')
print('Buró:',list(bureau.columns));print('Políticas:',list(pol.columns))
if bureau.empty or pol.empty:print('PENDIENTE: snapshots históricos de buró y políticas')

## Interacciones, selección de variables y linaje

Evaluar interacciones justificadas: $score\times TPV$, $utilización\times caída\_ventas$, $MOB\times campaña$, $sector\times región$. En modelos logísticos:
$$\operatorname{logit}(p_i)=\beta_0+\beta_1x_i+\beta_2z_i+\beta_3x_iz_i.$$
El efecto marginal de $x$ depende de $z$: $\partial p/\partial x=p(1-p)(\beta_1+\beta_3z)$. No interpretar correlación como causalidad.

In [ ]:
dictionary=load('diccionario_variables.csv')
print('Variables documentadas:',len(dictionary))
display(dictionary.head(12))

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.